# Cancer Risk Assistant

This chatbot is powered by two tools:

1. **Cancer Severity Prediction Project** – uses a Data Science model to predict a numerical cancer severity score from the cancer dataset.

2. **Guideline Search (WHO documents)** – retrieves relevant passages from a public-health document corpus using Retrieval-Augmented Generation (RAG), where each claim is supported by a citation.

As a result, the chatbot answers cancer-risk questions by routing each question to the appropriate source instead of letting the language model generate answers from its own knowledge.

The LLM (Qwen) never produces the risk score itself. Its job is to read the question, decide which tool(s) to call, extract the required arguments, and write the response based on what the tools return.

##1. Setup

Installing tools and API key

In [1]:
# This project requires a GPU to run the LLM.
# OR
# This project can also run on google colab by using T4 GPU for free

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is required. In Google Colab, go to "
        "Runtime > Change runtime type > select T4 GPU."
    )

print(f"Using GPU: {torch.cuda.get_device_name(0)}")

Using GPU: Tesla T4


In [2]:
!pip install -q sentence-transformers gradio scikit-learn joblib pypdf transformers accelerate bitsandbytes

print("Installing tool have finished")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.3 MB/s eta 0:00:00
Installing tool have finished


### LLM Model

I initially planned to use `Qwen` through `an external API`, such as OpenRouter. However, this would require an API key and users would need to sign in or provide their own credentials to run the chatbot. I was concerned that this would make the project less convenient for someone who wants to download and try the notebook.

I therefore chose to use `Qwen2.5-1.5B-Instruct` and load the model directly into the Google Colab environment instead. I previously tried other open-source LLMs such as LLaMA and Ollama during an academic project, but I found that Qwen performed better for my use case.

The model runs on the available Colab GPU, so no external LLM API or API key is required to run the chatbot. Users only need to enable a GPU runtime in Google Colab.

In [3]:
# Call Qwen to be an LLM model
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Qwen loaded successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded successfully!


## 2. Tool 1 - Cancer Severity Prediction Project

This tool uses the Linear Regression model selected from my self-study Data Science project to predict a cancer severity score based on the selected risk factors. I compared Linear Regression with a tuned Random Forest model and selected Linear Regression because it achieved better performance on the test set.

In [4]:
# Tool 1 - Cancer Severity Prediction
'''
This tool loads the Linear Regression model from the
Cancer_Severity_Prediction_Project and uses it to predict
a numerical cancer severity score from selected risk factors.

If a saved model is available, it will be loaded directly.
If not, the model will be retrained from the dataset and saved.

'''

# 1. import tools
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error



# 2. Model configuration
# Features used by the Linear Regression model
FEATURES = [
    "Age",
    "Genetic_Risk",
    "Air_Pollution",
    "Alcohol_Use",
    "Smoking",
    "Obesity_Level"
]

# Target variable predicted by the model
TARGET = "Target_Severity_Score"

# File paths
MODEL_PATH = Path("severity_model.joblib")
CSV_PATH = Path("global_cancer_patients_2015_2024.csv")



# 3. Load or train the model
# Load the saved model if it already exists
if MODEL_PATH.exists():

    model = joblib.load(MODEL_PATH)

    print("Loaded saved Linear Regression model.")

# If the saved model is not available, retrain it from the dataset
elif CSV_PATH.exists():

    print("Saved model not found. Training the model from the dataset...")

    # Load the cancer dataset
    df = pd.read_csv(CSV_PATH)

    # Select the features and target
    X = df[FEATURES]
    y = df[TARGET]

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    # Create a pipeline:
    # 1. Standardise the input features
    # 2. Train a Linear Regression model
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])

    # Train the model
    model.fit(X_train, y_train)

    # Make predictions on the test set
    predictions = model.predict(X_test)

    # Evaluate the model
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)

    print(f"Retrained model. R²={r2:.3f}  MAE={mae:.3f}")

    # Save the trained model for future use
    joblib.dump(model, MODEL_PATH)

    print(f"Model saved to: {MODEL_PATH}")


# If neither the model nor dataset is available
else:

    raise FileNotFoundError(
        "Please upload either 'severity_model.joblib' or "
        "'global_cancer_patients_2015_2024.csv' to the Colab environment."
    )



# 4. Extract model information
# Get the StandardScaler and Linear Regression model from the trained pipeline
scaler = model.named_steps["scaler"]
linear_model = model.named_steps["model"]

'''
Since the features are standardised, the coefficients can be
compared to see which features have stronger relationships
with the predicted severity score.
'''
COEFS = dict(zip(FEATURES, linear_model.coef_))

# Store the mean and standard deviation used by the scaler
FEATURE_MEANS = dict(zip(FEATURES, scaler.mean_))
FEATURE_STDS = dict(zip(FEATURES, scaler.scale_))


# Display the coefficients ranked by absolute value
print("\nFeature coefficients:")

display(
    pd.Series(COEFS)
    .sort_values(key=abs, ascending=False)
    .to_frame("coefficient")
    .round(3)
)



# 5. Prediction function

def predict_cancer_severity(
    age,
    genetic_risk,
    air_pollution,
    alcohol_use,
    smoking,
    obesity_level
):
    """
    Predict the cancer severity score using the trained
    Linear Regression model.

    The model expects six input features:
    Age, Genetic_Risk, Air_Pollution, Alcohol_Use,
    Smoking, and Obesity_Level.
    """

    # Create a DataFrame using the same feature names and order used when training the model.
    input_data = pd.DataFrame([{
        "Age": age,
        "Genetic_Risk": genetic_risk,
        "Air_Pollution": air_pollution,
        "Alcohol_Use": alcohol_use,
        "Smoking": smoking,
        "Obesity_Level": obesity_level
    }])

    # Use the trained pipeline to make the prediction.
    # The pipeline automatically applies the StandardScaler before using the Linear Regression model.
    prediction = model.predict(input_data)[0]

    # Return the predicted score as a regular Python float
    return round(float(prediction), 3)



# 6. Test the prediction tool
# Example input
example_score = predict_cancer_severity(
    age=50,
    genetic_risk=5,
    air_pollution=5,
    alcohol_use=5,
    smoking=5,
    obesity_level=5
)

print(f"\nExample predicted severity score: {example_score}")

Loaded saved Linear Regression model.

Feature coefficients:


,coefficient
Smoking,0.583
Genetic_Risk,0.580
Alcohol_Use,0.437
Air_Pollution,0.437
Obesity_Level,0.290
Age,0.000



Example predicted severity score: 4.951


### Wrapping the model as a callable tool

This function does more than just calling model.predict():

Fills in missing inputs with the dataset mean — users may not provide all six factors in their question.
Returns the contribution of each factor — so the chatbot can explain which factors are contributing more to the predicted score.
Runs a counterfactual prediction — for example, it can show what the predicted score would be if smoking was reduced to zero. This makes the result more useful and actionable.

In [5]:
def predict_severity(
    age=None,
    genetic_risk=None,
    air_pollution=None,
    alcohol_use=None,
    smoking=None,
    obesity_level=None
):
    '''
    Tool 1: Predict a cancer severity score (1-10) from risk factors.

    If the user does not provide a value for a factor,
    the dataset mean is used instead.

    The tool returns:
    - predicted severity score
    - contribution of each factor
    - counterfactual score if smoking is reduced to 0
    '''

    # Put all user inputs into a dictionary
    supplied = {
        "Age": age,
        "Genetic_Risk": genetic_risk,
        "Air_Pollution": air_pollution,
        "Alcohol_Use": alcohol_use,
        "Smoking": smoking,
        "Obesity_Level": obesity_level,
    }

    # If an input is missing (None), replace it with the mean value from the original dataset.
    # Otherwise, use the value provided by the user.
    used = {
        f: (FEATURE_MEANS[f] if v is None else float(v))
        for f, v in supplied.items()
    }

    # Keep track of which features were filled automatically
    defaults_used = [
        f for f, v in supplied.items()
        if v is None
    ]

    # Convert the input values into a DataFrame using the same feature order as the trained model.
    row = pd.DataFrame([used])[FEATURES]

    # Use the trained Linear Regression pipeline to predict the cancer severity score.
    score = float(model.predict(row)[0])

    # Calculate the contribution of each factor.
    # This shows how much each feature contributes to the prediction compared with its dataset mean.
    contributions = {
        f: float(
            COEFS[f]
            * (used[f] - FEATURE_MEANS[f])
            / FEATURE_STDS[f]
        )
        for f in FEATURES
    }

    # Start with no counterfactual result
    counterfactual = None

    # Only calculate the smoking counterfactual when the user's smoking value is above the dataset mean.
    if used["Smoking"] > FEATURE_MEANS["Smoking"]:

        # Make a copy of the original input so we can change only the smoking value.
        cf_row = row.copy()

        # Set smoking to 0 to simulate the "smoking reduced to 0" scenario.
        cf_row["Smoking"] = 0.0

        # Predict the new severity score under this scenario.
        counterfactual = {
            "scenario": "smoking reduced to 0",
            "new_score": round(
                float(model.predict(cf_row)[0]),
                2
            ),
        }

    # Return all the information that the LLM needs to generate the final response to the user.
    return {
        "severity_score": round(score, 2),

        # Explain what the score means
        "scale": "1-10, higher is more severe",

        # Show the actual values used by the model, including any values filled with dataset means.
        "inputs_used": {
            k: round(v, 2)
            for k, v in used.items()
        },

        # Show which inputs were missing and automatically filled.
        "defaults_used": defaults_used,

        # Return the factor contributions, sorted from largest to smallest absolute contribution.
        "contributions": {
            k: round(v, 3)
            for k, v in sorted(
                contributions.items(),
                key=lambda kv: abs(kv[1]),
                reverse=True
            )
        },

        # Return the counterfactual result.It will be None if the condition above was not met.
        "counterfactual": counterfactual,

        # Tell the LLM where the prediction comes from and the model's test-set performance.
        "source": (
            "Linear Regression model, "
            "R2=0.785 on held-out data "
            "(synthetic dataset)"
        ),
    }

'''
Test the tool with an example user input.
Only Age, Smoking, and Genetic Risk are provided.
The other three factors will automatically use
their dataset mean values.
'''
predict_severity(
    age=55,
    smoking=8,
    genetic_risk=7
)

{'severity_score': 5.96,
 'scale': '1-10, higher is more severe',
 'inputs_used': {'Age': 55.0,
  'Genetic_Risk': 7.0,
  'Air_Pollution': np.float64(5.0),
  'Alcohol_Use': np.float64(5.02),
  'Smoking': 8.0,
  'Obesity_Level': np.float64(4.99)},
 'defaults_used': ['Air_Pollution', 'Alcohol_Use', 'Obesity_Level'],
 'contributions': {'Smoking': 0.608,
  'Genetic_Risk': 0.401,
  'Age': 0.0,
  'Air_Pollution': 0.0,
  'Alcohol_Use': 0.0,
  'Obesity_Level': 0.0},
 'counterfactual': {'scenario': 'smoking reduced to 0', 'new_score': 4.34},
 'source': 'Linear Regression model, R2=0.785 on held-out data (synthetic dataset)'}

## 3. Tool 2 — Guideline Retrieval (RAG)

### About this knowledge base

This tool uses public-health information from official WHO and IARC webpages instead of a manually created sample knowledge base.

The code retrieves the content from the source webpages, processes the text, splits it into smaller chunks, and creates embeddings for retrieval. When a user asks a question, the most relevant sections are retrieved and passed to the LLM as context.

This allows the chatbot to answer questions using information from the original sources instead of relying only on the LLM's own knowledge.

The current sources include:

| Source                       | Link                                                                                      |
| ---------------------------- | ----------------------------------------------------------------------------------------- |
| WHO — Cancer fact sheet      | https://www.who.int/news-room/fact-sheets/detail/cancer                                   |
| WHO — Tobacco fact sheet     | https://www.who.int/news-room/fact-sheets/detail/tobacco                                  |
| WHO — Alcohol and cancer     | https://www.who.int/news-room/fact-sheets/detail/alcohol                                  |
| WHO — Obesity and overweight | https://www.who.int/news-room/fact-sheets/detail/obesity-and-overweight                   |
| WHO — Ambient air pollution  | https://www.who.int/news-room/fact-sheets/detail/ambient-(outdoor)-air-quality-and-health |
| WHO — Physical activity      | https://www.who.int/news-room/fact-sheets/detail/physical-activity                        |
| IARC Monographs              | https://monographs.iarc.who.int/                                                          |


In [6]:
# RAG Knowledge Base - Source URLs
'''
These are the official sources used by the chatbot.
The content will be downloaded from these webpages and then processed for retrieval.
'''

SOURCE_URLS = [
    {
        "doc": "WHO Cancer fact sheet",
        "url": "https://www.who.int/news-room/fact-sheets/detail/cancer",
    },
    {
        "doc": "WHO Tobacco fact sheet",
        "url": "https://www.who.int/news-room/fact-sheets/detail/tobacco",
    },
    {
        "doc": "WHO Alcohol fact sheet",
        "url": "https://www.who.int/news-room/fact-sheets/detail/alcohol",
    },
    {
        "doc": "WHO Obesity and overweight fact sheet",
        "url": "https://www.who.int/news-room/fact-sheets/detail/obesity-and-overweight",
    },
    {
        "doc": "WHO Ambient air pollution fact sheet",
        "url": "https://www.who.int/news-room/fact-sheets/detail/ambient-(outdoor)-air-quality-and-health",
    },
    {
        "doc": "WHO Physical activity fact sheet",
        "url": "https://www.who.int/news-room/fact-sheets/detail/physical-activity",
    },
    {
        "doc": "IARC Monographs overview",
        "url": "https://monographs.iarc.who.int/",
    },
]

print(f"{len(SOURCE_URLS)} knowledge base webpages have loaded.")

7 knowledge base webpages have loaded.


### 3.1 Chunking and Embedding

The source webpages are split into smaller text chunks so that the RAG system can search for the most relevant information.

Embedding converts each text chunk into a list of numbers. This allows the system to find text with similar meanings, even when the words are different. For example, a question like "how do I lower my risk" can still match information about "prevention strategies".

I use `all-MiniLM-L6-v2` because it is a small and free embedding model. It is lightweight enough to run efficiently in the Colab environment with GPU acceleration or the Local destop environment with CPU as well

If the embedding model cannot be downloaded, the code falls back to TF-IDF. TF-IDF is not as good at understanding similar meanings, but the rest of the RAG pipeline can still work.

In [7]:
# Load source webpages

import requests
from bs4 import BeautifulSoup


# Store the webpage content from each source.
DOCS = []


for source in SOURCE_URLS:

    try:
        # Download the webpage.
        response = requests.get(
            source["url"],
            timeout=20
        )

        # Stop if the webpage cannot be downloaded successfully.
        response.raise_for_status()

        # Parse the HTML content.
        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        # Remove elements that are not useful for retrieval.
        for element in soup(
            ["script", "style", "nav", "footer", "header"]
        ):
            element.decompose()

        # Extract the visible text from the webpage.
        text = soup.get_text(
            separator=" ",
            strip=True
        )

        # Store the document name, URL, and extracted text.
        DOCS.append(
            {
                "doc": source["doc"],
                "url": source["url"],
                "section": "Webpage",
                "text": text
            }
        )

        print(
            f"Loaded: {source['doc']} "
            f"({len(text):,} characters)"
        )

    except Exception as e:

        # Skip a source if the webpage cannot be loaded.
        print(
            f"Failed to load {source['doc']}: "
            f"{type(e).__name__}: {e}"
        )


# Show how many webpages were successfully loaded.
print(
    f"\nSuccessfully loaded "
    f"{len(DOCS)}/{len(SOURCE_URLS)} sources."
)

Loaded: WHO Cancer fact sheet (13,463 characters)
Loaded: WHO Tobacco fact sheet (11,853 characters)
Loaded: WHO Alcohol fact sheet (11,123 characters)
Loaded: WHO Obesity and overweight fact sheet (14,137 characters)
Loaded: WHO Ambient air pollution fact sheet (9,048 characters)
Loaded: WHO Physical activity fact sheet (11,249 characters)
Loaded: IARC Monographs overview (1,435 characters)

Successfully loaded 7/7 sources.


In [8]:
# Chunking and Embedding

# Create chunks from the documents downloaded from the WHO and IARC webpages.
# Each chunk keeps the original document name, URL, section, and text so that the chatbot can provide the source when answering the user.
CHUNKS = [
    {
        **d,
        "id": i,
        "citation": f"{d['doc']} — {d.get('section', 'Source')}"
    }
    for i, d in enumerate(DOCS)
]



# 1. Create embeddings

# EMBEDDER is used when the Sentence Transformer model is available.
EMBEDDER = None

# VECTORIZER is only used if the embedding model fails and the system needs to fall back to TF-IDF.
VECTORIZER = None


try:
    # Import the embedding model
    from sentence_transformers import SentenceTransformer

    # Load a small and efficient embedding model.
    # GPU is used here because the project is running in Colab.
    EMBEDDER = SentenceTransformer(
        "all-MiniLM-L6-v2",
        device="cuda"
    )

    # Convert each text chunk into an embedding vector.
    # normalize_embeddings=True makes similarity comparison easier.
    EMBEDDINGS = EMBEDDER.encode(
        [c["text"] for c in CHUNKS],
        normalize_embeddings=True,
        show_progress_bar=True
    )

    print(
        f"Embedded with all-MiniLM-L6-v2 -> "
        f"{EMBEDDINGS.shape}"
    )


except Exception as e:

    # If the embedding model cannot be loaded, use TF-IDF as a simpler fallback.
    from sklearn.feature_extraction.text import TfidfVectorizer

    print(
        f"sentence-transformers unavailable "
        f"({type(e).__name__}); using TF-IDF fallback."
    )

    # Create the TF-IDF vectorizer
    VECTORIZER = TfidfVectorizer(
        stop_words="english"
    )

    # Convert each text chunk into a TF-IDF vector
    EMBEDDINGS = VECTORIZER.fit_transform(
        [c["text"] for c in CHUNKS]
    ).toarray()

    # Normalize the vectors so they can be compared
    # using cosine similarity.
    EMBEDDINGS = (
        EMBEDDINGS
        / (
            np.linalg.norm(
                EMBEDDINGS,
                axis=1,
                keepdims=True
            ) + 1e-9
        )
    )

    print(
        f"TF-IDF matrix -> {EMBEDDINGS.shape}"
    )


# 2. Retrieval relevance cutoff

'''
Only retrieved chunks with a similarity score above this threshold should be considered relevant.

A higher threshold makes the system more selective and helps reduce the chance of giving the LLM irrelevant
information that could lead to hallucination.
'''

MIN_SCORE = (
    0.25
    if EMBEDDER is not None
    else 0.12
)

print(f"Relevance cutoff: {MIN_SCORE}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedded with all-MiniLM-L6-v2 -> (7, 384)
Relevance cutoff: 0.25


### 3.2 Retrieval

Since this project currently has only a small number of chunks, cosine similarity with a NumPy array is enough and runs very quickly. I do not use a vector database here because it would add extra dependencies and setup without much benefit at this scale. A vector database would make more sense if the knowledge base grows to tens of thousands of chunks.

The `min_score` is important for reducing irrelevant results. If none of the retrieved chunks are relevant enough, the tool returns **nothing** instead of forcing the system to use the closest match. This helps prevent the assistant from answering an unrelated question using an irrelevant source.

In [9]:
# 1. Retrieval


def _embed_query(q):
    '''
    Convert the user's question into an embedding vector
    using the same embedding model used for the documents.
    '''
    if EMBEDDER is not None:
        return EMBEDDER.encode(
            [q],
            normalize_embeddings=True
        )[0]

    # If the embedding model is unavailable, use the TF-IDF vectorizer as a fallback.
    v = VECTORIZER.transform([q]).toarray()[0]

    # Normalize the TF-IDF vector so that it can be compared with the document vectors.
    return v / (np.linalg.norm(v) + 1e-9)


def search_guidelines(query, top_k=3, min_score=None):
    """
    Tool 2: Retrieve relevant public-health information
    from the WHO and IARC knowledge base.

    Returns up to top_k relevant passages with their
    citation, source URL, and similarity score.

    If no passage reaches the minimum relevance score,
    the tool returns an empty result instead of using
    an unrelated passage.
    """

    # Use the default relevance cutoff unless a different value is provided.
    min_score = (
        MIN_SCORE
        if min_score is None
        else min_score
    )

    '''
    Convert the user's question into an embedding
    and compare it with all document embeddings.

    Because both vectors are normalized, the dot product
    gives the cosine similarity score.
    '''
    scores = EMBEDDINGS @ _embed_query(query)

    # Sort passages from highest to lowest similarity and keep only the top_k candidates.
    order = np.argsort(scores)[::-1][:top_k]

    '''
    Keep only passages that meet the minimum relevance score.

    This is important because the system should return no result when the available sources are not relevant
    enough, rather than forcing a weak match.
    '''
    hits = [
        {
            "text": CHUNKS[i]["text"],
            "citation": CHUNKS[i]["citation"],
            "url": CHUNKS[i]["url"],
            "score": round(float(scores[i]), 3),
        }
        for i in order
        if scores[i] >= min_score
    ]

    # Return the retrieved information to the LLM.
    return {
        "query": query,
        "n_results": len(hits),
        "results": hits
    }


# 2. Test the retrieval tool

# Example question about smoking cessation.
demo = search_guidelines(
    "how long after quitting smoking does risk go down"
)

# Display the retrieved passages and their similarity scores.
for h in demo["results"]:
    print(
        f"[{h['score']}] {h['citation']}\n"
        f"    {h['text'][:110]}...\n"
    )

[0.418] WHO Tobacco fact sheet — Webpage
    Tobacco and nicotine Skip to main content Tobacco and nicotine 26 June 2026 Reading time: Key facts Tobacco da...

[0.264] WHO Cancer fact sheet — Webpage
    Cancer Skip to main content Cancer 3 July 2026 Reading time: Key facts Cancer is a leading cause of death worl...



## 4. The Router — Deciding Which Tool to Call

The LLM is given both tools as JSON schemas and asked to return **only a JSON object** that specifies which tool to call and what arguments to use.

### Why use JSON instead of native function calling?

I chose to use JSON because it makes the routing logic simple and easy to see in this notebook. The LLM first decides what the user is asking for, then returns the tool name and the required arguments in a structured format.

The main trade-off is that the JSON output needs to be checked and validated before the tool is called. This helps prevent invalid tool calls and makes the routing process easier to test.

The project uses Qwen2.5-1.5B-Instruct locally in Google Colab, so no external LLM API or API key is required. If the local LLM cannot be loaded, the `rule_based_router` can be used as a simple fallback. It uses keyword matching to decide which tool to call, so it is less flexible than the LLM router but keeps the rest of the notebook runnable.

In [10]:
# Check whether the local Qwen model is available


# HAS_LLM is used by the router and compose functions to check whether the local Qwen model was loaded successfully.

try:
    HAS_LLM = (
        tokenizer is not None
        and model is not None
    )

except NameError:
    HAS_LLM = False


print(f"Local LLM available: {HAS_LLM}")

Local LLM available: True


In [11]:
# The Router

import json
import re


# 1. Tool instructions for the LLM

'''
Tell Qwen what tools are available and when to use them.
The model must return JSON only so that the routing decision can be checked before any tool is actually called.
'''

TOOL_SPEC = """
You are a router for a cancer information assistant.

You have two tools:

1. predict_severity(age, genetic_risk, air_pollution, alcohol_use, smoking, obesity_level)

   Use this ONLY when the user describes a specific person
   and provides personal risk information.

   All arguments are optional numbers.
   Omit any value that the user did not mention.

   age is in years.
   The other five factors use a 0-10 scale:
   0 = none
   10 = extreme

   Examples:
   "never smoked" = smoking 0
   "occasionally smoke" = smoking 3
   "heavy smoker" = smoking 8
   "family history of cancer" = genetic_risk 7


2. search_guidelines(query)

   Use this for general questions about:
   - cancer risk
   - cancer prevention
   - screening
   - smoking
   - alcohol
   - obesity
   - air pollution
   - physical activity
   - other cancer-related public-health information


Use both tools when the user asks about their own risk
AND asks what they should do.

If the question is unrelated to the purpose of this chatbot,
return no tool calls and mark it as out_of_scope.

For questions within scope, return:

{"calls": [{"tool": "<tool name>", "args": {...}}], "status": "ok"}

For questions outside the chatbot's scope, return:

{"calls": [], "status": "out_of_scope"}

Return ONLY valid JSON.

Do not include explanations, markdown, or any other text.
"""


# 2. Rule-based fallback router

def rule_based_router(question):
    """
    Simple fallback router.

    This is used if the local LLM is unavailable or
    if the LLM produces invalid output.
    """

    # Convert the question to lowercase to make keyword matching easier.
    q = question.lower()

    # Store the tools that should be called.
    calls = []

    # Check whether the user is talking about themselves.
    personal = any(
        w in q
        for w in [
            "i ",
            "i'm",
            "im ",
            "my ",
            "me ",
            "am i",
            "ผม",
            "ฉัน",
            "ดิฉัน",
            "ของฉัน",
        ]
    )

    # Extract numbers from the question.
    numbers = re.findall(r"\d+", q)


    # 3. Decide whether the prediction tool should be called

    if personal or numbers:

        args = {}

        # Treat a number between 18 and 100 as a possible age.
        for n in numbers:
            if 18 <= int(n) <= 100:
                args["age"] = int(n)
                break

        # Simple keyword mapping for smoking.
        if "smok" in q or "บุหรี่" in q:
            args["smoking"] = 8

        # Simple keyword mapping for alcohol.
        if (
            "drink" in q
            or "alcohol" in q
            or "เหล้า" in q
            or "แอลกอฮอล์" in q
        ):
            args["alcohol_use"] = 7

        # Simple keyword mapping for family/genetic risk.
        if (
            "family" in q
            or "father" in q
            or "mother" in q
            or "genetic" in q
            or "dad" in q
            or "mom" in q
        ):
            args["genetic_risk"] = 7

        # Only call the prediction tool if at least one relevant input was detected.
        if args:
            calls.append(
                {
                    "tool": "predict_severity",
                    "args": args
                }
            )


    # 4. Decide whether the guideline search tool should be called

    guideline_keywords = [
        "cancer",
        "cancers",
        "risk",
        "prevent",
        "prevention",
        "screening",
        "smoking",
        "smoke",
        "alcohol",
        "obesity",
        "pollution",
        "exercise",
        "physical activity",
        "tobacco",
        "มะเร็ง",
        "ความเสี่ยง",
        "ป้องกัน",
        "ตรวจคัดกรอง",
        "บุหรี่",
        "เหล้า",
        "แอลกอฮอล์",
    ]

    # Check whether the question contains a cancer/public-health related keyword.
    is_guideline_question = any(
        w in q
        for w in guideline_keywords
    )

    if is_guideline_question:
        calls.append(
            {
                "tool": "search_guidelines",
                "args": {
                    "query": question
                }
            }
        )


    # Return the routing plan.
    # If no relevant tool was selected, mark the question
    # as out_of_scope so the chatbot can give a safe response.
    if not calls:
        return {
            "calls": [],
            "status": "out_of_scope"
        }

    return {
        "calls": calls,
        "status": "ok"
    }


# 5. Local LLM router

def llm_router(question):
    """
    Ask the local Qwen model which tools to call.

    If the model is unavailable or produces invalid JSON,
    fall back to the rule-based router.
    """

    # If the local LLM was not loaded successfully, use the simpler keyword-based router.
    if not HAS_LLM:
        return rule_based_router(question)

    try:

        # Create the prompt for Qwen.
        messages = [
            {
                "role": "system",
                "content": TOOL_SPEC
            },
            {
                "role": "user",
                "content": question
            }
        ]

        # Convert the messages into the format expected by Qwen's chat template.
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Tokenize the prompt and move it to the GPU.
        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(llm_model.device)

        # Generate the routing decision.
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

        # Remove the original prompt tokens so that only the generated response remains.
        generated_tokens = outputs[
            0
        ][
            inputs["input_ids"].shape[1]:
        ]

        # Convert the generated tokens back to text.
        raw = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        # Remove markdown code fences if Qwen accidentally adds them.
        raw = re.sub(
            r"^```(?:json)?",
            "",
            raw,
            flags=re.IGNORECASE
        )

        raw = re.sub(
            r"```$",
            "",
            raw
        ).strip()

        # Convert the JSON text into a Python dictionary.
        plan = json.loads(raw)



        # 6. Validate the router output

        # The output must contain a list called "calls".
        if not isinstance(
            plan.get("calls"),
            list
        ):
            raise ValueError(
                "Router output does not contain a valid 'calls' list."
            )

        # Only allow the two tools that actually exist.
        valid_tools = {
            "predict_severity",
            "search_guidelines"
        }

        validated_calls = []

        for call in plan["calls"]:

            # Check that each call is a dictionary.
            if not isinstance(call, dict):
                continue

            # Get the requested tool name.
            tool_name = call.get("tool")

            # Ignore unknown tools.
            if tool_name not in valid_tools:
                continue

            # Make sure arguments are a dictionary.
            args = call.get("args", {})

            if not isinstance(args, dict):
                args = {}

            # If Qwen selects the prediction tool but fails
            # to extract any arguments, use the rule-based router.
            if (
                tool_name == "predict_severity"
                and not args
            ):
                raise ValueError(
                    "Prediction tool selected without arguments."
                )

            # Keep only valid tool calls.
            validated_calls.append(
                {
                    "tool": tool_name,
                    "args": args
                }
            )

        # Return the validated routing plan.
        # Keep the status returned by Qwen so that the chatbot
        # can distinguish between normal questions and
        # out-of-scope questions.
        return {
            "calls": validated_calls,
            "status": plan.get("status", "ok")
        }

    except Exception as e:

        # If anything goes wrong, use the rule-based router instead of stopping the rest of the notebook.
        print(
            f"[router fallback: {type(e).__name__}] {e}"
        )

        return rule_based_router(question)


# 7. Test the router

test_questions = [
    "I am 55, heavy smoker, my father had cancer. How risky am I?",
    "Does alcohol actually cause cancer?",
    "What is my risk and what should I do about it?",
    "What's the weather today?",
]

for q in test_questions:

    print(f"Q: {q}")

    # Run the router and display its JSON output.
    result = llm_router(q)

    print(
        "->",
        json.dumps(
            result,
            ensure_ascii=False
        )
    )

    print()

Q: I am 55, heavy smoker, my father had cancer. How risky am I?
[router fallback: ValueError] Prediction tool selected without arguments.
-> {"calls": [{"tool": "predict_severity", "args": {"age": 55, "smoking": 8, "genetic_risk": 7}}, {"tool": "search_guidelines", "args": {"query": "I am 55, heavy smoker, my father had cancer. How risky am I?"}}], "status": "ok"}

Q: Does alcohol actually cause cancer?
-> {"calls": [{"tool": "search_guidelines", "args": {"query": "alcohol causes cancer"}}], "status": "ok"}

Q: What is my risk and what should I do about it?
[router fallback: JSONDecodeError] Expecting ',' delimiter: line 1 column 179 (char 178)
-> {"calls": [{"tool": "search_guidelines", "args": {"query": "What is my risk and what should I do about it?"}}], "status": "ok"}

Q: What's the weather today?
[router fallback: JSONDecodeError] Expecting value: line 1 column 1 (char 0)
-> {"calls": [], "status": "out_of_scope"}



### Router Test Results

The router was tested with four example questions covering personal risk, general cancer information, combined questions, and out-of-scope questions.

The router correctly identified the appropriate tool for the alcohol question and correctly rejected the unrelated weather question. For the personal risk question, Qwen selected the prediction tool but returned empty arguments, so the validation step triggered the rule-based fallback. The fallback successfully extracted the relevant inputs: age 55, smoking level 8, and genetic risk 7.

The combined question was also handled by the fallback because Qwen produced invalid JSON. In this case, the fallback identified the question as a guideline-related query.

These results show that the routing layer can select the appropriate tools while maintaining a fallback mechanism when the local LLM produces incomplete or invalid structured output.

With the routing layer working, the next step is to **execute the selected tool calls and pass their results to the LLM to generate the final response**.

## 5. Executing the Plan and Composing the Reply

I split this part into two steps:

- **Execute** — Run the tools selected by the router. This part is handled entirely by Python, so the results do not depend on what the LLM says.
  
- **Compose** — Send the tool results back to the LLM and ask it to generate the final response. The LLM is instructed to use only the information returned by the tools, not make up any numbers or information, and keep the original citations.

Keeping these two steps separate makes the system easier to test. I can check whether the tools return the correct results independently from how the LLM writes the final response.

In [27]:
# Execute the Plan and Compose the Reply


# 1. Map each tool name to its Python function.
# The executor uses this dictionary to call the tool selected by the router.
TOOLS = {
    "predict_severity": predict_severity,
    "search_guidelines": search_guidelines
}


# The risk model needs at least this many personal details to be meaningful.
MIN_PERSONAL_DETAILS = 2


def execute(plan):
    """
    Run each call in the routing plan.

    The tools are executed by Python, not by the LLM.
    This keeps the tool results deterministic and
    independent from how the LLM writes the final answer.

    Returns a list containing:
    - tool name
    - arguments used
    - tool result
    """

    out = []

    # Go through each tool call selected by the router.
    for call in plan.get("calls", []):

        # Find the corresponding Python function.
        fn = TOOLS.get(call.get("tool"))

        # Ignore unknown tools.
        if fn is None:
            continue

        '''
        Guard against a risk score built mostly from dataset averages.

        The router sometimes selects the risk model for a general
        knowledge question such as "how long after quitting smoking
        does risk drop", and extracts the word "smoking" as if it were
        a personal detail. With only one value supplied, every other
        input would be a dataset mean, so the score would say almost
        nothing about the user. In that case the pipeline records that
        more personal details are needed instead of running the model.
        '''
        if call.get("tool") == "predict_severity":

            supplied = [
                v for v in call.get("args", {}).values()
                if v is not None
            ]

            if len(supplied) < MIN_PERSONAL_DETAILS:

                out.append(
                    {
                        "tool": call["tool"],
                        "args": call.get("args", {}),
                        "result": {"needs_input": True}
                    }
                )

                continue

        try:

            # Run the selected tool with the arguments provided by the router.
            result = fn(**call.get("args", {}))

        except TypeError as e:

            # Return an error if the router provided invalid arguments for the tool.
            result = {
                "error": f"bad arguments: {e}"
            }

        # Store the tool name, arguments, and result so they can be passed to the LLM later.
        out.append(
            {
                "tool": call["tool"],
                "args": call.get("args", {}),
                "result": result
            }
        )

    return out


# 2. Instructions for composing the final answer

'''
The rules are split into a base block and two tool-specific blocks.

Only the rules for the tools that actually ran are sent to the LLM.
This prevents the model from inventing a severity score or
contributing factors for a question that never called the risk model.
'''

COMPOSE_BASE = """
You are a cancer risk assistant.

Write the final reply using ONLY the CONTEXT provided below.

Rules:

- Do not add any fact, number, factor, or cause that is not
  in the CONTEXT.

- Do not provide a medical diagnosis or personalised
  medical advice.

- Do not write citations, URLs, or source names.
  They are added automatically after your reply.

- Answer in 3 to 5 sentences. No headings, no bullet lists,
  no numbered lists.

- If the CONTEXT does not answer the question, say so plainly
  instead of guessing.
"""

COMPOSE_RULES_SCORE = """
- The severity score is already shown to the user, so do not
  repeat the number. Explain in plain words which factors raise
  or lower the score, and what would change if smoking stopped.

- Never use internal words such as "counterfactual", "context",
  or "contribution value".
"""

COMPOSE_RULES_GUIDELINE = """
- Summarise only what the guideline passages say. Do not mention
  a severity score or contributing factors, because the risk model
  was not used for this question.
"""

# The details the risk model needs before it can produce a useful score.
DETAILS_NEEDED = (
    "your age, how heavily you smoke, how much alcohol you drink, "
    "your air pollution exposure, and your obesity level"
)

# Shown when the risk model was asked for, but too few details were given.
NEEDS_INPUT_MESSAGE = (
    f"To estimate your severity score I need a few details first: "
    f"{DETAILS_NEEDED}.\n\n"
    "For example: *\"I'm 55, I smoke heavily and I drink occasionally.\"*"
)

# Shown when neither tool produced anything usable.
NO_RESULT_MESSAGE = (
    "I could not find enough information to answer this reliably. "
    "If you are asking about your own risk, tell me "
    f"{DETAILS_NEEDED}, and I can calculate a severity score for you."
)


# Plain-English labels for the model's feature names.
FEATURE_LABELS = {
    "Age": "age",
    "Genetic_Risk": "genetic risk",
    "Air_Pollution": "air pollution",
    "Alcohol_Use": "alcohol use",
    "Smoking": "smoking",
    "Obesity_Level": "obesity level",
}


def _build_context(executed):
    """
    Turn the raw tool results into a short, clean CONTEXT block.

    Sending compact text instead of the raw JSON stops the small
    model from copying internal field names such as 'counterfactual'
    into the reply.

    The severity score and the citations are kept separately so that
    Python can print them itself. The LLM can then never change a
    number or invent a source.

    Returns:
    - context text for the LLM
    - header text printed by Python (severity score block)
    - list of (citation, url) pairs
    - set of tool names that produced usable output
    - whether the risk model still needs personal details
    """

    blocks = []
    header = ""
    citations = []
    tools_used = set()
    needs_input = False

    for step in executed:

        r = step["result"]

        # Skip tool calls that failed.
        if not isinstance(r, dict) or "error" in r:
            continue

        # The risk model was skipped because too few details were given.
        if r.get("needs_input"):
            needs_input = True
            continue

        # Handle the cancer severity prediction result.
        if step["tool"] == "predict_severity":

            tools_used.add(step["tool"])

            # Select the three largest contributing factors.
            top = list(
                r["contributions"].items()
            )[:3]

            # The score is printed by Python, not by the LLM.
            header = (
                f"**Severity score: {r['severity_score']} / 10**  \n"
                f"_{r['scale']}_"
            )

            # Tell the user which values were guessed from dataset averages.
            if r.get("defaults_used"):

                assumed = ", ".join(
                    FEATURE_LABELS.get(f, f)
                    for f in r["defaults_used"]
                )

                header += (
                    f"  \n_Assumed dataset averages for: {assumed}. "
                    f"Provide these details for a more accurate result._"
                )

            text = (
                f"Severity score: {r['severity_score']} out of 10.\n"

                + "How each factor moves the score:\n"

                + "\n".join(
                    f"- {FEATURE_LABELS.get(k, k)}: {v:+.2f}"
                    for k, v in top
                )
            )

            if r.get("counterfactual"):
                text += (
                    f"\nIf this person stopped smoking, the score would "
                    f"be about {r['counterfactual']['new_score']}."
                )

            blocks.append(text)

        # Handle the guideline search result.
        else:

            for hit in r.get("results", []):

                tools_used.add(step["tool"])

                blocks.append(
                    f"Guideline passage: {hit['text']}"
                )

                citations.append(
                    (hit["citation"], hit["url"])
                )

    return "\n\n".join(blocks), header, citations, tools_used, needs_input


def compose(question, executed):
    """
    Generate the final response using the tool results.

    The LLM is only responsible for rewriting the CONTEXT
    into readable English. It does not calculate the severity
    score, retrieve guideline information, or write citations.
    """

    # If no tools were executed, the question was outside the scope of the chatbot.
    if not executed:
        return (
            "Sorry, I can't help with this question because "
            "it is outside the scope of this chatbot. This "
            "chatbot is designed to provide information about "
            "cancer risk, severity prediction, and prevention."
        )

    # Build the compact context, the score header, and the citations.
    context, header, citations, tools_used, needs_input = _build_context(executed)

    # Nothing usable was produced, but the risk model can run once details are given.
    if not context.strip() and needs_input:
        return NEEDS_INPUT_MESSAGE

    # Neither tool produced anything usable.
    if not context.strip():
        return NO_RESULT_MESSAGE

    '''
    Citations are attached by Python as real Markdown links,
    so the user can click through to the original source and
    the LLM can never alter them.
    '''
    seen = dict(citations)

    sources = (
        "\n\n" + "\n".join(
            f"📄 [{c}]({u})"
            for c, u in seen.items()
        )
        if citations
        else ""
    )

    # The score block always comes first, printed by Python.
    prefix = header + "\n\n" if header else ""

    '''
    Guideline information was found, but the user still has to supply
    personal details before a severity score can be calculated.
    '''
    note = (
        f"\n\n_To also calculate your severity score, tell me "
        f"{DETAILS_NEEDED}._"
        if needs_input
        else ""
    )


    # 3. Deterministic fallback if the local LLM is unavailable

    if not HAS_LLM:
        return prefix + context + sources + note


    # 4. Use the local Qwen model to compose the final reply

    # Send only the rules that match the tools that actually ran.
    system_prompt = COMPOSE_BASE

    if "predict_severity" in tools_used:
        system_prompt += COMPOSE_RULES_SCORE
    else:
        system_prompt += COMPOSE_RULES_GUIDELINE

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": (
                f"Question: {question}\n\n"
                f"CONTEXT:\n{context}"
            )
        }
    ]

    # Convert the messages into the format expected by Qwen.
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize the prompt and move it to the GPU.
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm_model.device)

    # Generate the final response.
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=220,
        do_sample=False
    )

    # Remove the original prompt tokens.
    generated_tokens = outputs[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]

    # Convert the generated tokens back into text.
    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # Python prints the score and the citations, the LLM only explains.
    return prefix + answer + sources + note


# 5. Main chatbot function

def ask(question, show_trace=False):
    """
    Run the complete chatbot pipeline:

    User question
    → Router
    → Execute tools
    → Compose final answer
    """

    # Step 1: Ask the router which tools should be used.
    plan = llm_router(question)

    # Step 2: Execute the selected tools.
    executed = execute(plan)

    # Step 3: Compose the final answer using the tool results.
    answer = compose(
        question,
        executed
    )

    # Optional debugging information.
    if show_trace:

        print(
            "PLAN:",
            json.dumps(
                plan,
                ensure_ascii=False
            ),
            "\n"
        )

        for step in executed:

            print(
                f"TOOL {step['tool']}({step['args']}) ->",
                json.dumps(
                    step["result"],
                    ensure_ascii=False
                )[:220],
                "\n"
            )

    return answer



# 6. Test the complete pipeline

print(
    ask(
        "I'm 55, I smoke heavily and my father had cancer. "
        "What is my risk and what should I do?",
        show_trace=True
    )
)

PLAN: {"calls": [{"tool": "predict_severity", "args": {"age": 55, "genetic_risk": 7, "air_pollution": 0, "alcohol_use": 0, "smoking": 8, "obesity_level": 0}}], "status": "ok"} 

TOOL predict_severity({'age': 55, 'genetic_risk': 7, 'air_pollution': 0, 'alcohol_use': 0, 'smoking': 8, 'obesity_level': 0}) -> {"severity_score": 3.95, "scale": "1-10, higher is more severe", "inputs_used": {"Age": 55.0, "Genetic_Risk": 7.0, "Air_Pollution": 0.0, "Alcohol_Use": 0.0, "Smoking": 8.0, "Obesity_Level": 0.0}, "defaults_used": [], "co 

**Severity score: 3.95 / 10**  
_1-10, higher is more severe_

Your risk for developing cancer is higher due to heavy smoking. Stopping smoking can significantly reduce your risk.


## 6. Chat Interface

I use Gradio to create a simple chat interface that runs directly in Google Colab.

The interface allows users to interact with the chatbot without needing to run the functions manually. I also use `share=True` to generate a temporary public link, which makes it easier to access the chatbot from another device and record the demo.

In [28]:
!pip install -q -U gradio # for using CSS

In [29]:
import gradio as gr

# User Interface


DISCLAIMER = (
    "⚠️ Educational prototype. The risk model is trained on a "
    "synthetic Kaggle dataset and is not clinically validated. "
    "Guideline information is retrieved from official WHO and IARC "
    "sources. Not medical advice."
)



# 1. Example Questions
EXAMPLES = [
    "I'm 55, I smoke heavily and my father had cancer. How risky am I?",
    "How long after quitting smoking does lung cancer risk drop?",
    "What is my risk and what should I do about it?",
    "Does alcohol really cause cancer?",
]



# 2. Response Function
# Uses the real ask() pipeline defined in section 5.

def respond(message, history):

    if not message or not message.strip():
        yield "", history
        return

    history = history or []


    # 2.1. Immidiately shown user's question and representing a waiting box (Processing...)
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": "⏳ Processing..."}
    ]
    yield "", history


    # 2.2. AI Pipeline
    try:
        answer = ask(message)
    except Exception as e:
        answer = f"⚠️ Sorry, something went wrong while generating the answer.\n\n`{e}`"

    if answer is None:
        answer = "⚠️ No answer was returned."


    # 2.3. Provide an exact respond of AI after finished processing and replace a processing box
    history[-1]["content"] = str(answer)
    yield "", history



# 3. Custom CSS - Web Design

CUSTOM_CSS = """

/* ============================================================
   PAGE
   ============================================================ */
.gradio-container {
    background: #F4F8F6 !important;
    max-width: 1150px !important;
    margin: auto !important;
}

/* ============================================================
   HEADER
   ============================================================ */
#app-header {
    text-align: center !important;
    padding: 25px 20px 15px 20px !important;
}
#app-header h1 {
    color: #163F3F !important;
    font-size: 32px !important;
    font-weight: 700 !important;
    margin: 0 !important;
}
#app-header p {
    color: #657B76 !important;
    font-size: 15px !important;
    margin-top: 7px !important;
}

/* ============================================================
   CHAT WINDOW
   ============================================================ */
#chat-window {
    background: #FFFFFF !important;
    border: 1px solid #DCE8E3 !important;
    border-radius: 18px !important;
    box-shadow: 0 4px 20px rgba(22, 63, 63, 0.06) !important;
    min-height: 430px !important;
}

/* ============================================================
   USER MESSAGE
   ============================================================ */
#chat-window .message.user {
    background: #176B6B !important;
    color: #FFFFFF !important;
    border-radius: 16px 16px 5px 16px !important;
    padding: 11px 15px !important;
}
#chat-window .message.user * {
    color: #FFFFFF !important;
}

/* ============================================================
   ASSISTANT MESSAGE
   ============================================================ */
#chat-window .message.bot {
    background: #EEF6F3 !important;
    color: #244444 !important;
    border: 1px solid #DCEAE5 !important;
    border-radius: 16px 16px 16px 5px !important;
    padding: 12px 16px !important;
}
#chat-window .message.bot * {
    color: #244444 !important;
}

/* ============================================================
   TRY ASKING
   ============================================================ */
#example-title {
    color: #49645F !important;
    font-size: 13px !important;
    font-weight: 600 !important;
    margin: 15px 0 8px 3px !important;
}

/* ============================================================
   EXAMPLE BUTTONS
   ============================================================ */
.example-btn {
    background: #FFFFFF !important;
    color: #176B6B !important;
    border: 1px solid #CFE0DA !important;
    border-radius: 12px !important;
    min-height: 65px !important;
    padding: 12px 15px !important;
    font-size: 13px !important;
    font-weight: 500 !important;
    text-align: left !important;
    transition: 0.15s ease !important;
}
.example-btn span {
    color: #176B6B !important;
}
.example-btn:hover {
    background: #EAF5F1 !important;
    border-color: #176B6B !important;
}

/* ============================================================
   INPUT AREA
   ============================================================ */
#input-container {
    background: #FFFFFF !important;
    border: 1px solid #CFE0DA !important;
    border-radius: 17px !important;
    padding: 5px !important;
    margin-top: 14px !important;
    box-shadow: 0 5px 20px rgba(22, 63, 63, 0.08) !important;
}

/* ============================================================
   TEXTBOX
   ============================================================ */
#message-box textarea {
    background: #FFFFFF !important;
    color: #244444 !important;
    border: none !important;
    box-shadow: none !important;
    font-size: 15px !important;
    padding: 12px 14px !important;
}
#message-box textarea::placeholder {
    color: #91A5A0 !important;
}
#message-box textarea:focus {
    border: none !important;
    box-shadow: none !important;
}

/* ============================================================
   SEND BUTTON
   ============================================================ */
#send-button {
    background: #35A66F !important;
    color: #FFFFFF !important;
    border: none !important;
    border-radius: 12px !important;
    width: 50px !important;
    min-width: 50px !important;
    height: 50px !important;
    min-height: 50px !important;
    font-size: 21px !important;
    font-weight: 700 !important;
    margin: auto 3px auto 0 !important;
}
#send-button:hover {
    background: #2E9363 !important;
}
#send-button span {
    color: #FFFFFF !important;
}

/* ============================================================
   DISCLAIMER
   ============================================================ */
#disclaimer {
    text-align: center !important;
    color: #7A8C87 !important;
    font-size: 11px !important;
    margin-top: 10px !important;
}

/* ============================================================
   HIDE FOOTER
   ============================================================ */
footer {
    display: none !important;
}
"""



# 4. Build UI

with gr.Blocks() as demo_ui:


    # 4.1. Header
    gr.HTML(
        """
        <div id="app-header">
        <h1>🩺 Cancer Risk Assistant</h1>
        <p>Data-driven cancer severity & prevention insights</p>
        </div>
        """
    )



    # 4.2. Chatbot
    chatbot = gr.Chatbot(
        elem_id="chat-window",
        height=430,
        show_label=False,
    )



    # 4.3. Try Asking
    gr.Markdown(
        "Try asking",
        elem_id="example-title",
    )

    with gr.Row():
        example_1 = gr.Button(EXAMPLES[0], elem_classes="example-btn")
        example_2 = gr.Button(EXAMPLES[1], elem_classes="example-btn")

    with gr.Row():
        example_3 = gr.Button(EXAMPLES[2], elem_classes="example-btn")
        example_4 = gr.Button(EXAMPLES[3], elem_classes="example-btn")



    # 4.4. Input
    with gr.Row(elem_id="input-container"):
        message_box = gr.Textbox(
            placeholder="Ask about cancer risk, severity, or prevention...",
            show_label=False,
            lines=1,
            scale=10,
            elem_id="message-box",
        )
        send_button = gr.Button(
            "➤",
            elem_id="send-button",
            scale=0,
        )



    # 4.5. Disclaimer
    gr.Markdown(DISCLAIMER, elem_id="disclaimer")



    # 4.6. SEND BUTTON
    send_button.click(
        respond,
        inputs=[message_box, chatbot],
        outputs=[message_box, chatbot],
    )



    # 4.7. ENTER KEY = SEND
    message_box.submit(
        respond,
        inputs=[message_box, chatbot],
        outputs=[message_box, chatbot],
    )



    # 4.8. TRY ASKING BUTTONS
    example_1.click(lambda: EXAMPLES[0], inputs=None, outputs=message_box)
    example_2.click(lambda: EXAMPLES[1], inputs=None, outputs=message_box)
    example_3.click(lambda: EXAMPLES[2], inputs=None, outputs=message_box)
    example_4.click(lambda: EXAMPLES[3], inputs=None, outputs=message_box)


# 5. Launch
if __name__ == "__main__":

    # Close any existing Gradio server before launching a new one.
    gr.close_all()

    demo_ui.launch(
        share=True,
        debug=False,
        css=CUSTOM_CSS,
    )

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bdd068e82673eafc5a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 7. Evaluation

I use two tests to check if the main parts of the system are working properly. The third claim cannot be tested because the training data is synthetic.

| Claim | Testable? | How |
|---|---|---|
| The assistant reports the model's number correctly | ✅ | 7.1 — compare the score in the assistant's reply with the model's original output |
| The retrieval system finds the correct source | ✅ | 7.2 — test questions where I already know which source should be retrieved |
| The score represents real cancer risk | ❌ | Cannot be tested because the training data is synthetic. See Section 8. |



### 7.1 Numeric Fidelity

I tested 10 cases to check if the assistant reports the model's score correctly.

For each case, I first run the input directly through the model to get the original severity score. I then run the same input through the full assistant and check if the assistant gives the same score.

If the scores are different, it means the LLM changed the number instead of using the actual result from the model.


In [30]:
# Numeric Fidelity Test Cases

# 10 test cases with different combinations of risk factors.
FIDELITY_CASES = [

    {"age": 55, "smoking": 8, "genetic_risk": 7},

    {"age": 30, "smoking": 0, "genetic_risk": 2},

    {"age": 70, "smoking": 9, "alcohol_use": 8, "obesity_level": 6},

    {"age": 45, "air_pollution": 9},

    {"age": 25, "smoking": 1, "alcohol_use": 1, "obesity_level": 1},

    {"age": 60, "genetic_risk": 9},

    {"age": 50, "smoking": 5, "alcohol_use": 5},

    {"age": 65, "obesity_level": 8, "air_pollution": 6},

    {"age": 35, "smoking": 7},

    {"age": 80, "genetic_risk": 3, "smoking": 2, "alcohol_use": 4},

]


# Store the results of each test case.
rows = []


# Run each case through the prediction model and then through the assistant.
for case in FIDELITY_CASES:

    # Get the original score directly from the prediction model.
    expected = predict_severity(**case)["severity_score"]

    # Create the same tool call that the router would generate.
    plan = {
        "calls": [
            {
                "tool": "predict_severity",
                "args": case
            }
        ],
        "status": "ok"
    }

    # Run the tool and generate the assistant's reply.
    reply = compose(
        "What is my risk?",
        execute(plan)
    )

    # Find decimal numbers reported in the assistant's reply.
    found = [
        float(x)
        for x in re.findall(
            r"\d+\.\d+",
            reply
        )
    ]

    # Check whether the model's original score appears in the assistant's response.
    ok = any(
        abs(v - expected) < 0.05
        for v in found
    )

    # Store the result for this test case.
    rows.append(
        {
            "case": str(case)[:46],
            "model_score": expected,
            "in_reply": "yes" if ok else "NO",
            "pass": ok
        }
    )


# Convert the test results into a DataFrame.
fidelity = pd.DataFrame(rows)


# Count how many test cases passed.
passed = fidelity["pass"].sum()


# Print the overall result.
print(
    f"Numeric fidelity: "
    f"{passed}/{len(fidelity)} passed\n"
)


# Display the results without the internal pass column.
fidelity.drop(
    columns="pass"
)

Numeric fidelity: 10/10 passed



,case,model_score,in_reply
0,"{'age': 55, 'smoking': 8, 'genetic_risk': 7}",5.96,yes
1,"{'age': 30, 'smoking': 0, 'genetic_risk': 2}",3.34,yes
2,"{'age': 70, 'smoking': 9, 'alcohol_use': 8, 'o",6.32,yes
3,"{'age': 45, 'air_pollution': 9}",5.56,yes
4,"{'age': 25, 'smoking': 1, 'alcohol_use': 1, 'o",3.14,yes
5,"{'age': 60, 'genetic_risk': 9}",5.76,yes
6,"{'age': 50, 'smoking': 5, 'alcohol_use': 5}",4.95,yes
7,"{'age': 65, 'obesity_level': 8, 'air_pollution",5.41,yes
8,"{'age': 35, 'smoking': 7}",5.36,yes
9,"{'age': 80, 'genetic_risk': 3, 'smoking': 2, '",3.79,yes



### 7.2 Retrieval Accuracy

I tested 10 questions where I already knew which source should contain the relevant information. I then checked if the retrieval system found the expected source.

This test only checks whether the retriever can find the right source. It does not check whether the LLM uses the retrieved information correctly in its final answer.

The last two cases are **negative cases**. The knowledge base does not have relevant information for these questions, so the correct behaviour is to return no results. This checks that the retriever does not simply return a source for every question, even when the information is not relevant.

In [31]:
# Retrieval Accuracy Test Cases


# Ten questions with a known expected source document.
# The last two are negative cases where the knowledge base
# should not return any relevant result.
RETRIEVAL_CASES = [

    ("How long after quitting does lung cancer risk fall?", "Tobacco"),

    ("Is alcohol a carcinogen?", "Alcohol"),

    ("Does being overweight cause cancer?", "Obesity"),

    ("Can air pollution cause lung cancer?", "Ambient air"),

    ("How much exercise should adults get?", "Physical activity"),

    ("What share of cancers can be prevented?", "Cancer fact sheet"),

    ("Does the HPV vaccine prevent cancer?", "Cancer fact sheet"),

    ("What does IARC Group 1 mean?", "IARC"),

    ("What is the capital of Thailand?", None),

    ("How do I fix a flat tyre?", None),

]


# Store the results of each retrieval test.
rows = []


# Run each question through the retrieval system.
for question, expected_doc in RETRIEVAL_CASES:

    # Search the knowledge base for relevant passages.
    res = search_guidelines(question)

    # Get the highest-scoring result if one was returned.
    top = (
        res["results"][0]
        if res["results"]
        else None
    )



    # Check negative cases

    if expected_doc is None:

        # For negative cases, the correct behaviour is to return no result.
        ok = top is None

        got = (
            "(nothing returned)"
            if top is None
            else top["citation"]
        )



    # Check expected source

    else:

        # Check whether the top retrieved result comes from the expected source document.
        ok = (
            top is not None
            and expected_doc.lower()
            in top["citation"].lower()
        )

        got = (
            top["citation"]
            if top
            else "(nothing returned)"
        )


    # Store the result of this test case.
    rows.append(
        {
            "question": question[:44],
            "expected": expected_doc or "(none)",
            "retrieved": got[:44],
            "pass": ok
        }
    )


# Convert the results into a DataFrame.
retrieval = pd.DataFrame(rows)


# Print the overall retrieval accuracy.
print(
    f"Retrieval accuracy: "
    f"{retrieval['pass'].sum()}/{len(retrieval)} passed\n"
)


# Display the detailed test results.
retrieval

Retrieval accuracy: 6/10 passed



,question,expected,retrieved,pass
0,How long after quitting does lung cancer ris,Tobacco,WHO Cancer fact sheet — Webpage,False
1,Is alcohol a carcinogen?,Alcohol,WHO Alcohol fact sheet — Webpage,True
2,Does being overweight cause cancer?,Obesity,WHO Cancer fact sheet — Webpage,False
3,Can air pollution cause lung cancer?,Ambient air,WHO Cancer fact sheet — Webpage,False
4,How much exercise should adults get?,Physical activity,WHO Physical activity fact sheet — Webpage,True
5,What share of cancers can be prevented?,Cancer fact sheet,WHO Cancer fact sheet — Webpage,True
6,Does the HPV vaccine prevent cancer?,Cancer fact sheet,WHO Cancer fact sheet — Webpage,True
7,What does IARC Group 1 mean?,IARC,(nothing returned),False
8,What is the capital of Thailand?,(none),(nothing returned),True
9,How do I fix a flat tyre?,(none),(nothing returned),True


## 8. Results and Limitations

### 8.1 What the system does

Four question types were tested through the chat interface. The pipeline
routes each one differently:

| Question type | Example | Behaviour |
|---|---|---|
| General knowledge | *"Does alcohol really cause cancer?"* | Retrieves WHO/IARC passages, summarises them, and lists clickable source links |
| Personal risk, enough detail | *"I'm 60, I smoke heavily and I also drink alcohol"* | Returns the severity score, states which inputs were assumed, explains the contributing factors, and cites the relevant guidelines |
| Personal risk, too little detail | *"What is my risk and what should I do about it?"* | Asks for the missing details instead of producing a score built from dataset averages |
| Out of scope | *"What's the capital of Thailand?"* | Declines and states what the chatbot is for |

### 8.2 End-to-end testing through the chat interface

Sections 7.1 and 7.2 tested the two tools in isolation. This section reports
manual end-to-end testing through the Gradio interface, where the router, the
tools, and the composer all run together and the user only sees the final reply.

Eight questions were entered, covering the four routing paths above.

| # | Question entered in the UI | Expected behaviour | Observed result | Verdict |
|---|---|---|---|---|
| 1 | *Does alcohol really cause cancer?* | Guideline retrieval only | Summary of WHO passages with three clickable source links, no severity score | Pass |
| 2 | *Can air pollution increase the risk of lung cancer?* | Guideline retrieval only | Correct summary with three source links, but included figures ("4.2 million deaths in 2019", "89%") not present in the retrieved passages | Partial |
| 3 | *How long after quitting smoking does lung cancer risk drop?* | Guideline retrieval only | No misleading score was shown; the system asked for personal details instead of answering from the guidelines | Partial |
| 4 | *I'm 55, I smoke heavily and my father had cancer. How risky am I?* | Risk model + guidelines | Score 5.96 / 10, assumed inputs listed, three source links | Pass |
| 5 | *I'm 60, I smoke heavily and I also drink alcohol. What does my severity score look like, and what does WHO recommend about these risk factors?* | Both tools in one turn | Score 5.86 / 10 with the counterfactual, plus tobacco and alcohol source links | Pass |
| 6 | *I'm 55, I smoke heavily and I drink occasionally.* | Risk model | Score 2.99 / 10 returned correctly, but the explanation described a negative contribution as increasing risk | Partial |
| 7 | *What is my risk and what should I do about it?* | Request missing details | Asked for age, smoking, alcohol, air pollution and obesity level rather than scoring from dataset averages | Pass |
| 8 | *What's the capital of Thailand?* | Refuse | Declined and stated the scope of the chatbot | Pass |

Five of eight cases behaved exactly as designed. The three partial cases are all
related to limitations of the small LLM, but they occur in different parts of the pipeline.

### 8.3 Design decision: what Python owns and what the LLM owns

The severity score, the list of assumed inputs, and every citation are printed
by Python. The LLM receives a short context block and is only asked to turn it
into readable English. It never sees raw JSON field names, and it is explicitly
forbidden from writing citations.

This split means the severity score shown to the user always comes from the regression
model, while source links always come from the retrieval tool. Neither can be
altered by the language model.

Testing through the UI confirmed the practical effect: across all eight turns,
no severity score contradicted the regression model, and no citation pointed anywhere
other than a document the retrieval tool actually returned. Where the system failed,
the severity score and source links remained correct, but the explanatory prose was
sometimes inaccurate or unsupported.

### 8.4 Limitations and why they exist

The composer is Qwen2.5-1.5B-Instruct, chosen so the project runs end to end on
a free Colab T4 GPU. A model of that size follows long rule sets only loosely,
and all three partial cases trace back to that constraint.

**Ungrounded statistics in free text (case 2).** The model fills gaps from its
own pretraining when the retrieved passage is thinner than the question. The
citations remain correct, which arguably makes this the most dangerous failure
mode: the invented figure inherits the credibility of a real WHO link.

**Imperfect routing (case 3).** The router extracted "smoking" from *"quitting
smoking"* and selected the risk model for a question that was never about the
user. A guard in `execute()` blocks the risk model when fewer than two personal
details are supplied, so no meaningless score reaches the user, but the question
is still not redirected to the guideline tool as it should be.

**Misreading the sign of a contribution value (case 6).** A contribution of
−1.01 lowers the score, and the model described it as increasing risk. Small
models reason poorly over signed numbers embedded in text.

**Small knowledge base.** The corpus is a handful of WHO and IARC pages with a
cosine-similarity cutoff. Questions inside the general topic but outside those
pages return no passage rather than a weak match. This is deliberate — a wrong
citation is worse than no answer — but it limits coverage.

**Synthetic training data.** The regression model reaches R² = 0.785 on
held-out data, but that data is a synthetic Kaggle dataset. The score
demonstrates the tool-calling architecture; it is not a clinically meaningful
number.

The main limitations are in the LLM-based routing and composition layers rather
than the deterministic prediction tool itself. These parts of the system are not
deterministic. A larger instruction-following model would address the first three;
further prompt engineering reduces their frequency but does not remove them.

### 8.5 Conclusion

The architecture holds: deterministic tools produce the facts, and the language
model is confined to presentation. Five of eight end-to-end cases passed
outright, and in the three partial cases the system degraded safely — it asked
for more information or showed correct numbers with an imperfect explanation,
rather than inventing a risk score. The remaining defects are concentrated in
the free-tier language model, which is the expected trade-off when the entire
system is built on free resources.